### Cell 1: Load and inspect

Load the Part 2 sample (500 students, 43,574 interactions) and the EdNet-Content `questions.csv`, and check shapes, columns, and a few rows before joining.

In [1]:
import pandas as pd 

sample = pd.read_csv('../data/raw/kt1_sample.csv')
questions = pd.read_csv('../data/raw/questions.csv')

print(sample.shape, sample.columns.tolist())
print(questions.shape, questions.columns.tolist())

sample.head(), questions.head()

(43574, 6) ['timestamp', 'solving_id', 'question_id', 'user_answer', 'elapsed_time', 'user_id']
(13169, 7) ['question_id', 'bundle_id', 'explanation_id', 'correct_answer', 'part', 'tags', 'deployed_at']


(       timestamp  solving_id question_id user_answer  elapsed_time user_id
 0  1498906740412           1        q129           b         27000  u22094
 1  1498906764080           2       q8058           b         22000  u22094
 2  1498906786376           3       q8120           a         20000  u22094
 3  1498906818824           4        q157           d         30000  u22094
 4  1498906848124           5         q52           b         27000  u22094,
   question_id bundle_id explanation_id correct_answer  part          tags  \
 0          q1        b1             e1              b     1   1;2;179;181   
 1          q2        b2             e2              a     1      15;2;182   
 2          q3        b3             e3              b     1  14;2;179;183   
 3          q4        b4             e4              b     1   9;2;179;184   
 4          q5        b5             e5              c     1   8;2;179;181   
 
      deployed_at  
 0  1558093217098  
 1  1558093219720  
 2  155809322

**What this tells us:** `sample` is a pure event log — one row per interaction (`timestamp`, `question_id`, `user_answer`, `elapsed_time`, `user_id`) with no correctness or skill information. `questions` is the other half: one row per question, with `correct_answer` and `part` (the skill), but nothing about who answered it. Neither table alone can compute correctness or a per-skill mastery signal — that's why the join in the next cell is necessary. The gap in row counts (43,574 interactions vs. 13,169 distinct questions) also shows the sample draws repeatedly from a much larger, shared question pool rather than each student seeing unique questions.

### Cell 2: Join and compute correctness

Left-join `sample` onto `questions` on `question_id` to attach `correct_answer` and `part` (the skill label). Drop rows with a missing `user_answer` or an unmatched question, then derive `correct = (user_answer == correct_answer)`. Sort by `user_id`, then `timestamp`, so each student's interactions are in chronological order — required for BKT.

In [2]:
data = sample.merge(
    questions[["question_id", "correct_answer", "part"]],
    on="question_id",
    how="left",
)

print("missing user_answer:", data["user_answer"].isna().sum())
data = data.dropna(subset=["user_answer", "correct_answer", "part"]).copy()

data["correct"] = (data["user_answer"] == data["correct_answer"]).astype(int)

data = data.sort_values(["user_id", "timestamp"], kind="mergesort").reset_index(drop=True)

print(data.shape)
print(data["correct"].mean())

data[["user_id", "timestamp", "question_id", "part", "correct"]].head()

missing user_answer: 4
(43570, 9)
0.6245122790911177


,user_id,timestamp,question_id,part,correct
0,u10054,1510480965027,q8098,1,1
1,u10054,1510480991061,q8074,1,0
2,u10054,1510481027410,q176,1,0
3,u10054,1510481048942,q1279,2,0
4,u10054,1510481129281,q2067,3,0


**What this tells us:** only 4 of 43,574 rows had no recorded answer — negligible, and safe to drop. Row count barely changed after the join (43,574 → 43,570), meaning almost every interaction matched a question in the metadata, so join coverage is essentially complete. The overall correctness rate, **62.45%**, is the base rate students get right across all skills combined; it's the number a trained BKT model's per-skill predictions should be compared against (and roughly what a "predict the average" baseline would score, relevant to README's E1).

### Sanity check

Confirm the join/drop didn't lose more rows than expected, and look at how interactions are distributed across skills (`part`) and students.

In [3]:
print("rows before/after:", len(sample), len(data))
print(data["part"].value_counts().sort_index())
print("students:", data["user_id"].nunique())

rows before/after: 43574 43570
part
1     3634
2     7307
3     2637
4     2922
5    21725
6     3935
7     1410
Name: count, dtype: int64
students: 500


**What this tells us:** the row count confirms exactly 4 rows were lost, matching the missing-`user_answer` count — nothing else was dropped unexpectedly. All 500 students are still present, so no student was fully removed. The skill distribution is heavily imbalanced: `part` 5 alone accounts for ~50% of all interactions (21,725 rows), while `part` 7 has only ~3% (1,410 rows). BKT will have far more data to fit skill 5's parameters than skill 7's, so mastery estimates and any AUC results for the rarer skills should be treated as less reliable.

### Cell 3: Split into train and test

Split by `user_id` rather than by row, so each student's full interaction history stays on one side of the split. This avoids leaking a student's other interactions across train/test, and matches the held-out-student evaluation described in the README (E1).

In [4]:
from sklearn.model_selection import train_test_split

train_users_ids, test_users_ids= train_test_split(
    data["user_id"].unique(), test_size=0.2, random_state=42
)

train_df = data[data['user_id'].isin(train_users_ids)].reset_index(drop=True)
test_df = data[data['user_id'].isin(test_users_ids)].reset_index(drop=True)

print(f"train: {train_df['user_id'].nunique()} -> students {len(train_df)} rows")
print(f"test: {test_df['user_id'].nunique()} -> students {len(test_df)} rows")


train: 400 -> students 31454 rows
test: 100 -> students 12116 rows


**What this tells us:** the 500 students split into 400 train / 100 test, exactly the 80/20 target at the student level. But the *row* split isn't 80/20 (31,454 train vs. 12,116 test rows, ≈72%/28%) — because interactions per student are heavily skewed (README: median 10, max 6,332), a handful of high-volume students can shift the row split away from the student-level ratio. That's expected and fine, since the split target is students, not rows.

### Cell 4: Save train and test

Write `train_df` and `test_df` to `data/processed/` so `02_train_bkt.ipynb` can load them without re-running this notebook.

In [5]:
import os

os.makedirs("../data/processed", exist_ok=True)

train_df.to_csv("../data/processed/train.csv", index=False)
test_df.to_csv("../data/processed/test.csv", index=False)

print("saved:", len(train_df), "train rows,", len(test_df), "test rows")

saved: 31454 train rows, 12116 test rows
